# Data Preparation — GSE216139 (Mouse Stomach)

Load DGE matrices for mouse stomach samples from GSE216139.
Samples include antrum (adult_6um) and corpus (Corpus_K, Corpus_T) regions.

**Source**: [GSE216139](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE216139)  
**Format**: Tab-separated DGE text files (genes x cells)  
**Species**: Mouse

## 1. Setup & Imports

In [1]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import csr_matrix

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(6, 4))

from importlib.metadata import version
print(f"Scanpy: {version('scanpy')}  |  AnnData: {version('anndata')}  |  NumPy: {np.__version__}  |  Pandas: {pd.__version__}")

Scanpy: 1.10.3  |  AnnData: 0.10.9  |  NumPy: 1.26.4  |  Pandas: 2.2.3


## 2. Project Configuration

In [2]:
PROJECT_NAME = 'gastric_antrum'
GEO_ID = 'GSE216139'

PROJECT_DIR = Path('..')
RAW_DIR = Path('RAW')
OUT_DIR = PROJECT_DIR / 'data' / 'processed'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sample metadata: (filename, sample_id, region)
# adult_6um = antrum, Corpus_K/T = corpus
import glob
raw_files = sorted(RAW_DIR.glob('*.txt.gz'))

SAMPLES = []
for f in raw_files:
    name = f.name
    # Extract GSM ID as sample_id
    sample_id = name.split('_')[0]
    # Determine region from filename
    if 'adult_6um' in name:
        region = 'antrum'
    elif 'Corpus' in name:
        region = 'corpus'
    else:
        region = 'unknown'
    SAMPLES.append((name, sample_id, region))

print(f"Project: {PROJECT_NAME}")
print(f"Dataset: {GEO_ID}")
print(f"Output:  {OUT_DIR}")
print(f"\nSamples: {len(SAMPLES)}")
for name, sid, region in SAMPLES:
    print(f"  {sid}  {region:8s}  {name}")

Project: gastric_antrum
Dataset: GSE216139
Output:  ../data/processed

Samples: 18
  GSM6659199  antrum    GSM6659199_adult_6um_tube1_p23_S5_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659200  antrum    GSM6659200_adult_6um_tube1_p24_S3_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659201  antrum    GSM6659201_adult_6um_tube1_p25_S6_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659202  antrum    GSM6659202_adult_6um_tube2_p29_S4_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659203  antrum    GSM6659203_adult_6um_tube2_p30_S7_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659204  antrum    GSM6659204_adult_6um_tube2_p31_S8_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659205  corpus    GSM6659205_Corpus_K_PID00166_S2_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659206  corpus    GSM6659206_Corpus_K_PID00167_S4_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659207  corpus    GSM6659207_Corpus_K_PID00168_S6_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659208  corpus    GSM6659208_Corpus_K_PID00169_S8_mouse_mouse_seqlev_d2_dge.txt.gz
  GSM6659209  c

## 3. Load Data

In [3]:
adatas = {}

for filename, sample_id, region in SAMPLES:
    filepath = RAW_DIR / filename
    print(f"\nLoading {sample_id} ({region})...")

    # Read DGE: genes x cells -> transpose to cells x genes
    df = pd.read_csv(filepath, sep='\t', index_col=0).T
    print(f"  {df.shape[0]} cells x {df.shape[1]} genes")

    adata = ad.AnnData(
        X=csr_matrix(df.values),
        obs=pd.DataFrame(index=df.index),
        var=pd.DataFrame(index=df.columns),
    )

    adata.obs['sample_id'] = sample_id
    adata.obs['dataset'] = GEO_ID
    adata.obs['species'] = 'mouse'
    adata.obs['region'] = region
    adata.obs['condition'] = 'WT'

    for col in ['sample_id', 'dataset', 'species', 'region', 'condition']:
        adata.obs[col] = adata.obs[col].astype('category')

    adata.var_names_make_unique()
    adatas[sample_id] = adata

print(f"\nLoaded {len(adatas)} samples.")


Loading GSM6659199 (antrum)...


  384 cells x 19955 genes

Loading GSM6659200 (antrum)...


  384 cells x 19107 genes

Loading GSM6659201 (antrum)...


  384 cells x 18443 genes

Loading GSM6659202 (antrum)...


  384 cells x 20861 genes

Loading GSM6659203 (antrum)...


...

  384 cells x 21024 genes

Loading GSM6659213 (corpus)...


  384 cells x 20979 genes

Loading GSM6659214 (corpus)...


  384 cells x 21010 genes

Loading GSM6659215 (corpus)...


  384 cells x 20690 genes

Loading GSM6659216 (corpus)...


  384 cells x 21054 genes

Loaded 18 samples.


## 4. QC Metrics

In [4]:
for sample_id, adata in adatas.items():
    adata.var['mt'] = adata.var_names.str.startswith('mt-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

n_mt = adatas[list(adatas.keys())[0]].var['mt'].sum()
print(f"Mitochondrial genes detected: {n_mt} (prefix 'mt-')")

Mitochondrial genes detected: 24 (prefix 'mt-')


## 5. Summary

In [5]:
summary = []
for sample_id, adata in adatas.items():
    summary.append({
        'sample_id': sample_id,
        'region': adata.obs['region'].iloc[0],
        'n_cells': adata.n_obs,
        'n_genes': adata.n_vars,
        'median_genes': int(np.median(adata.obs['n_genes_by_counts'])),
        'median_counts': int(np.median(adata.obs['total_counts'])),
        'median_pct_mt': round(np.median(adata.obs['pct_counts_mt']), 2),
    })

summary_df = pd.DataFrame(summary)
print(f"{'='*85}")
print(f"GSE216139 Summary")
print(f"{'='*85}")
print(summary_df.to_string(index=False))
print(f"{'='*85}")
print(f"Total cells: {summary_df['n_cells'].sum():,}")
print(f"  Antrum: {summary_df.loc[summary_df['region']=='antrum', 'n_cells'].sum():,}")
print(f"  Corpus: {summary_df.loc[summary_df['region']=='corpus', 'n_cells'].sum():,}")

GSE216139 Summary
 sample_id region  n_cells  n_genes  median_genes  median_counts  median_pct_mt
GSM6659199 antrum      384    19955           123           1495          60.57
GSM6659200 antrum      384    19107           119           1139          73.08
GSM6659201 antrum      384    18443           122           1459          74.84
GSM6659202 antrum      384    20861           235           3080          49.26
GSM6659203 antrum      384    19857           123           1974          72.78
GSM6659204 antrum      384    19867           126           1983          68.32
GSM6659205 corpus      384    21505          6058          27863           5.66
GSM6659206 corpus      384    21132          6127          27952           5.36
GSM6659207 corpus      384    20642          6014          28075           5.53
GSM6659208 corpus      384    20766          5803          25410           5.44
GSM6659209 corpus      384    20772          5886          27371           5.30
GSM6659210 corpus     

## 6. Save

In [6]:
# Save per-sample files
for sample_id, adata in adatas.items():
    out_path = OUT_DIR / f'{sample_id}_raw.h5ad'
    adata.write(out_path)
    print(f"Saved {out_path.name} ({adata.n_obs} cells)")

# Save combined file
combined = ad.concat(adatas, label='sample_key', join='outer', fill_value=0)
combined.obs_names_make_unique()
combined_path = OUT_DIR / f'{GEO_ID}_combined_raw.h5ad'
combined.write(combined_path)
print(f"\nSaved {combined_path.name} ({combined.n_obs} cells x {combined.n_vars} genes)")

Saved GSM6659199_raw.h5ad (384 cells)
Saved GSM6659200_raw.h5ad (384 cells)


Saved GSM6659201_raw.h5ad (384 cells)


Saved GSM6659202_raw.h5ad (384 cells)
Saved GSM6659203_raw.h5ad (384 cells)


Saved GSM6659204_raw.h5ad (384 cells)


Saved GSM6659205_raw.h5ad (384 cells)


...

Saved GSM6659214_raw.h5ad (384 cells)


Saved GSM6659215_raw.h5ad (384 cells)


Saved GSM6659216_raw.h5ad (384 cells)


/Users/zahra/miniconda3/envs/bioinfo/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")



Saved GSE216139_combined_raw.h5ad (6912 cells x 31201 genes)


In [7]:
# Verify
test = sc.read_h5ad(combined_path)
print(f"Verification: {combined_path.name}: {test.shape[0]} cells x {test.shape[1]} genes  OK")

Verification: GSE216139_combined_raw.h5ad: 6912 cells x 31201 genes  OK
